In [ ]:
%run _bootstrap_dev.ipynb

In [ ]:
#
# 1 - portafoglio da analizzare
#

my_portfolio=greta_base_spy_portfolio_etf_ita        
my_portfolio_title="Gretchen S&P 500 Core-Engine - Fiduciaria"          # Start: 2016-01-04 CAGR: 13.39% Max DD: 22.03%   (SPY Only -> 12.40% 21.45%)  ( SWDA -> 10.53% 21.55%)    (QQQ -> 15.72% 25.42%)  

# my_portfolio=robohuman_portfolio
# my_portfolio_title="Robohuman"

# my_portfolio=greta_hy_portfolio 
# my_portfolio_title="Gretchen HY"          # Start: 2016-01-04 CAGR: 13.99% Max DD: 23.66%  

# my_portfolio=greta_base_bitcoin_portfolio
# my_portfolio_title="Gretchen Base Bitcoin"  # Start: 2021-04-26 CAGR: 13.09% Max DD: 23.73% 

# my_portfolio=greta_hy_bitcoin_portfolio
# my_portfolio_title="Gretchen HY Bitcoin"    # Start: 2021-04-26 CAGR: 13.78% Max DD: 24.48%

benchmark = 'SPY'

# my_portfolio=global_benchmark
# my_portfolio_title="Global Base"

# my_portfolio=core2_benchmark
# my_portfolio_title="Core2 Benchmark"

# my_portfolio=robohuman_portfolio
# my_portfolio_title="Robotica & Umanoidi"


# my_portfolio=PTF_NO_OVERLAP_IEF
# my_portfolio=PTF_NO_OVERLAP_SHY
# my_portfolio_title="Core-Satellite Patrimoniale 40"



# my_portfolio=greta_alt_A    #  CAGR: 11.49%  Max DD: 22.25% 
# my_portfolio=greta_alt_B    #  CAGR: 11.91%  Max DD: 22.35% 
# my_portfolio=greta_alt_C    #  CAGR: 10.63%  Max DD: 17.55%  
# my_portfolio_title="Gretchen C Base"         



# my_tickers=list(my_portfolio.keys())
# my_weights=list(my_portfolio.values())

#
# 2 - periodi di analisi
#

# periodo di analisi/backtest
# start_date = '2014-01-01'
# start_date = '2016-01-04'

start_date = None
end_date = None

# start_date = '2019-10-01'
# end_date = '2025-08-01'

# 3 - commissioni e capitale (BH -> commissioni non incidono)
init_cash=166_000
fees=0.0

In [ ]:
# my_tickers=list(my_portfolio.keys())
# my_weights=list(my_portfolio.values())

# normalize=True

# if normalize:
#     stocks_data, company_data, common_start_date, common_end_date = fetch_data_and_companies(my_tickers, start_date, end_date,normalize=normalize)
#     start_date = common_start_date
#     end_date = common_end_date
# else:
#     stocks_data, company_data = fetch_data_and_companies(my_tickers, start_date, end_date)

# # stocks_data
# company_info = company_data.loc[company_data.index.intersection(my_tickers)]
# my_display(stocks_data)
# my_display(company_info)

## Multifrequencies backtests

In [ ]:
# %run mc_functions.ipynb

# Step 1: test frequenza di bilanciamento
rebalance_freqs=['W', 'M', 'Q', 'Y', None]

for rebalance_freq in rebalance_freqs:
    # Esegui il backtest 
    final_portfolio  = run_bh_backtest(my_portfolio, 
                                       start_date=start_date, 
                                       end_date=end_date, 
                                       init_cash=init_cash, 
                                       fees=fees,
                                       rebalance_freq=rebalance_freq)
    
    # Stampa il risultato
    print("Backtest completato")
    print(f"\n--- Statistiche del Portafoglio {BOLD}{my_portfolio_title}{RESET} (freq: {rebalance_freq}) ---")
    # print(final_portfolio.stats())
    # final_portfolio.plot(group_by=True,width=vbt_plot_width).show()
    fig_value = final_portfolio.plot(
        width=vbt_plot_width,
        subplots=[
            'cum_returns',
            'drawdowns',
            'underwater',
            ]
    )
    fig_value.show()
    _ = print_summary(final_portfolio)

In [ ]:
# # benchmark_data = download_data(benchmark,start_date,end_date)

# figs_all = generate_lazy_portfolio_performance(final_portfolio,
#                                                      my_portfolio_title, 
#                                                      benchmark=benchmark,
#                                                      benchmark_data=benchmark_data)

## Preferred portfolio backtest

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _compute_weights_from_pf(pf) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calcola weights e asset_values (incluso CASH) da un vectorbt.Portfolio."""
    if pf is None:
        raise ValueError("pf is None")

    qty = pf.assets()
    if isinstance(qty, pd.Series):
        qty = qty.to_frame()

    prices = pf.close
    if isinstance(prices, pd.Series):
        prices = prices.to_frame()

    qty, prices = qty.align(prices, join="inner", axis=0)
    qty, prices = qty.align(prices, join="inner", axis=1)

    asset_values = qty * prices

    if hasattr(pf, "value"):
        total_value = pf.value().reindex(asset_values.index)
    else:
        total_value = asset_values.sum(axis=1)

    weights = asset_values.div(total_value, axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if hasattr(pf, "cash"):
        cash = pf.cash().reindex(asset_values.index).fillna(0.0)
        weights["CASH"] = (cash / total_value).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        asset_values["CASH"] = cash

    return weights, asset_values


def _collapse_weights_row(w_row: pd.Series, top_n: int = 12, keep: tuple[str, ...] = ("CASH",)) -> pd.Series:
    """Collassa un vettore pesi in Top-N + OTHER, mantenendo sempre le voci in `keep` se presenti."""
    w = w_row.copy()
    w = w[w.abs() > 0]

    forced = pd.Series(dtype=float)
    for k in keep:
        if k in w.index:
            forced.loc[k] = w.loc[k]
            w = w.drop(index=k)

    w = w.sort_values(ascending=False)
    top = w.iloc[:max(top_n, 0)]
    other = w.iloc[max(top_n, 0):].sum()

    out = pd.concat([top, forced])
    if other > 0:
        out.loc["OTHER"] = other

    return out.sort_values(ascending=False)


def _snapshot_yearly(weights: pd.DataFrame, shift_trading_days: int = 2, current_label: str = "CURRENT") -> pd.DataFrame:
    """
    Snapshot pesi all'inizio di ogni anno (shift di N trading days) + CURRENT (ultimo giorno disponibile).
    """
    idx = weights.index
    years = pd.Index(idx.year).unique()

    snap_dates = []
    for y in years:
        idx_year = idx[idx.year == y]
        if len(idx_year) == 0:
            continue
        pos = min(shift_trading_days, len(idx_year) - 1)
        snap_dates.append(idx_year[pos])

    snap = weights.loc[snap_dates].copy()
    snap.index = pd.Index([d.year for d in snap.index], name="Year")

    last_date = weights.index[-1]
    current_row = weights.loc[[last_date]].copy()
    current_row.index = pd.Index([current_label], name="Year")

    return pd.concat([snap, current_row], axis=0)


def _cash_metrics(cash_w: pd.Series, thresholds=(0.005, 0.01, 0.02, 0.05)) -> pd.DataFrame:
    """Metriche sintetiche su CASH weight (0..1)."""
    cw = cash_w.dropna().astype(float)
    stats = {
        "Cash mean (%)": cw.mean() * 100,
        "Cash median (%)": cw.median() * 100,
        "Cash max (%)": cw.max() * 100,
        "Cash p90 (%)": cw.quantile(0.90) * 100,
        "Cash p95 (%)": cw.quantile(0.95) * 100,
        "Cash p99 (%)": cw.quantile(0.99) * 100,
    }

    for th in thresholds:
        stats[f"Days cash > {th*100:.1f}%"] = int((cw > th).sum())
        stats[f"Pct days cash > {th*100:.1f}%"] = (cw > th).mean() * 100

    # streak >1%
    arr = (cw > 0.01).to_numpy(dtype=bool)
    best = cur = 0
    for v in arr:
        if v:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    stats["Max consecutive days cash > 1.0%"] = best

    df = pd.DataFrame.from_dict(stats, orient="index", columns=["Value"])
    df["Value"] = df["Value"].astype(float).round(3)
    return df


def _cash_by_year(cash_w: pd.Series) -> pd.DataFrame:
    """Tabella annuale: media/max cash% per anno + CURRENT (ultimo valore come max)."""
    cw = cash_w.dropna().astype(float)
    df = pd.DataFrame({"cash_w": cw})
    df["year"] = df.index.year

    out = df.groupby("year")["cash_w"].agg(["mean", "max"])
    out = out.rename(columns={"mean": "Cash mean (%)", "max": "Cash max (%)"})
    out["Cash mean (%)"] = (out["Cash mean (%)"] * 100).round(2)
    out["Cash max (%)"] = (out["Cash max (%)"] * 100).round(2)
    out.index.name = "Year"

    current = pd.DataFrame(
        {"Cash mean (%)": [np.nan], "Cash max (%)": [round(cw.iloc[-1] * 100, 2)]},
        index=pd.Index(["CURRENT"], name="Year"),
    )
    return pd.concat([out, current], axis=0)


def visualize_portfolio_weights(
    pf,
    title="Portfolio Weights",
    show_report: bool = True,
    vbt_plot_width: int | None = None,
    auto_threshold: int = 12,
    top_n: int = 12,
    shift_trading_days: int = 2,
    fig_height: int = 1150
):
    """
    Auto-mode:
    - Se tickers <= auto_threshold: stacked area full.
    - Altrimenti: stacked area compressa (Top-N + CASH + OTHER).

    Report:
    - Tabella snapshot annuali (inizio anno + CURRENT) via my_display.
    - Cash metrics (tabella) + Cash by Year (tabella) via my_display.

    Figure (unica):
    - Row 1: stacked weights evolution
    - Row 2: ONE pie (allocation) con dropdown per selezione anno/CURRENT
    - Row 3: CASH (%) over time
    """
    try:
        weights, _ = _compute_weights_from_pf(pf)
    except Exception:
        return None
    if weights.empty:
        return None

    last_date = weights.index[-1]
    n_cols = weights.shape[1]
    full_mode = n_cols <= auto_threshold

    # Snapshot annuali (+ CURRENT)
    snap = _snapshot_yearly(weights, shift_trading_days=shift_trading_days, current_label="CURRENT")

    # --- Tabella snapshots (auto-mode) ---
    if full_mode:
        snap_table = (snap * 100).round(2)
    else:
        collapsed_rows = []
        for idx_row in snap.index:
            row_c = _collapse_weights_row(snap.loc[idx_row], top_n=top_n, keep=("CASH",))
            row_c.name = idx_row
            collapsed_rows.append(row_c)
        snap_c = pd.DataFrame(collapsed_rows).fillna(0.0)
        snap_c.index.name = "Year"
        snap_table = (snap_c * 100).round(2)

        # Area chart: Top-N per peso medio (escludendo CASH)
        mean_w = weights.drop(columns=["CASH"], errors="ignore").mean().sort_values(ascending=False)
        top_cols = list(mean_w.iloc[:top_n].index)

        cols_to_keep = top_cols.copy()
        if "CASH" in weights.columns:
            cols_to_keep.append("CASH")

        w_sub = weights[cols_to_keep].copy()
        other_cols = [c for c in weights.columns if c not in cols_to_keep]
        if len(other_cols) > 0:
            w_sub["OTHER"] = weights[other_cols].sum(axis=1)

        w_sub = w_sub.div(w_sub.sum(axis=1), axis=0).fillna(0.0)
        weights_plot = w_sub

    if full_mode:
        weights_plot = weights

    # --- Cash tables ---
    if show_report and "CASH" in weights.columns:
        my_display(snap_table, title=f"{title} - Yearly Snapshots")
        my_display(_cash_metrics(weights["CASH"]), title=f"{title} - Cash Metrics")
        my_display(_cash_by_year(weights["CASH"]), title=f"{title} - Cash by Year (mean/max)")
    elif show_report:
        my_display(snap_table, title=f"{title} - Yearly Snapshots")

    # --- Build pie data per label (year/CURRENT) ---
    pie_labels = list(snap.index)  # e.g. [2019, 2020, ..., 'CURRENT']
    pie_data = {}

    for lbl in pie_labels:
        row = snap.loc[lbl]
        if not full_mode:
            row = _collapse_weights_row(row, top_n=top_n, keep=("CASH",))
        else:
            row = row[row.abs() > 0].sort_values(ascending=False)
        pie_data[str(lbl)] = (row.index.astype(str).tolist(), row.values.tolist())

    # Default pie = CURRENT (se presente), altrimenti ultimo label
    default_lbl = "CURRENT" if "CURRENT" in pie_data else str(pie_labels[-1])
    default_pie_labels, default_pie_values = pie_data[default_lbl]

    # --- Figure 3x1 ---
    fig = make_subplots(
        rows=3, cols=1,
        specs=[[{"type": "xy"}],
               [{"type": "domain"}],
               [{"type": "xy"}]],
        row_heights=[0.55, 0.25, 0.20],
        vertical_spacing=0.14,
        subplot_titles=(
            f"{title} — Weights evolution" + ("" if full_mode else f" (Top {top_n} + OTHER)"),
            f"{title} — Allocation (select year) — showing: {default_lbl}",
            f"{title} — CASH (%) over time"
        )
    )

    # Row 1: stacked area
    for col in weights_plot.columns:
        fig.add_trace(
            go.Scatter(
                x=weights_plot.index,
                y=weights_plot[col],
                mode="lines",
                stackgroup="one",
                name=str(col),
                hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2%}<extra>" + str(col) + "</extra>"
            ),
            row=1, col=1
        )
    fig.update_yaxes(title_text="Weight", tickformat=".0%", row=1, col=1)

    # Row 2: single pie (will be updated by dropdown)
    fig.add_trace(
        go.Pie(
            labels=default_pie_labels,
            values=default_pie_values,
            hole=0.45,
            sort=False,
            textinfo="label+percent",
            showlegend=False
        ),
        row=2, col=1
    )

    # Row 3: CASH line
    if "CASH" in weights.columns:
        fig.add_trace(
            go.Scatter(
                x=weights.index,
                y=weights["CASH"] * 100,
                mode="lines",
                name="CASH (%)",
                hovertemplate="%{x|%Y-%m-%d}<br>%{y:.2f}%<extra>CASH</extra>"
            ),
            row=3, col=1
        )
        fig.update_yaxes(title_text="CASH (%)", ticksuffix="%", row=3, col=1)
    else:
        # if no cash, keep empty axis clean
        fig.update_yaxes(visible=False, row=3, col=1)
        fig.update_xaxes(visible=False, row=3, col=1)

    # Dropdown to switch pie
    # Pie trace is at index: len(weights_plot.columns) (0-based) because we added that many scatters first
    pie_trace_index = len(weights_plot.columns)

    buttons = []
    for lbl in pie_data.keys():
        lab, vals = pie_data[lbl]
        buttons.append(
            dict(
                label=str(lbl),
                method="update",
                args=[
                    {"labels": [lab], "values": [vals]},  # applies to the pie trace
                    {"annotations": None}  # keep annotations stable (we update title text below)
                ],
                args2=None
            )
        )

    # Instead of rewriting annotations list (fragile), update the figure title text (stable)
    # We attach a second updatemenu with relayout to update subplot title row2 via layout.annotations index=1
    # Plotly makes subplot_titles into layout.annotations in order; row2 title is usually annotation index=1.
    # We'll compute safely: get current annotations count and target index=1.
    # If your environment changes annotation order, set pie_title_annotation_index accordingly.
    pie_title_annotation_index = 1

    # Replace buttons with a combined update+relayout using "args" + "method":"update" can't change annotations reliably.
    # So we build relayout buttons instead (labels/values via update, title via relayout) using method="update"
    # with layout update that targets the correct annotation.
    fixed_buttons = []
    for lbl in pie_data.keys():
        lab, vals = pie_data[lbl]
        fixed_buttons.append(
            dict(
                label=str(lbl),
                method="update",
                args=[
                    {"labels": [lab], "values": [vals]},
                    {"annotations": [
                        # we keep other annotations unchanged by using the existing ones at runtime
                    ]}
                ]
            )
        )

    # We cannot safely inject the entire annotations list here without reading fig.layout.annotations,
    # so we do the simplest robust approach: just show selected label in the dropdown itself.
    # (Row2 subplot title already states "select year".)
    fig.update_layout(
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                x=0.5,
                xanchor="center",
                y=0.53,           # between row1 and row2
                yanchor="top",
                buttons=[
                    dict(
                        label=str(lbl),
                        method="update",
                        args=[{"labels": [pie_data[str(lbl)][0]], "values": [pie_data[str(lbl)][1]]},
                              {}]
                    )
                    for lbl in pie_data.keys()
                ]
            )
        ]
    )

    # Layout sizing
    fig.update_layout(
        autosize=False if vbt_plot_width is not None else True,
        width=int(vbt_plot_width) if vbt_plot_width is not None else None,
        height=int(fig_height),
        title=dict(text=title, x=0.5),
        hovermode="x unified",
        legend_title_text="Ticker",
        margin=dict(l=60, r=60, t=120, b=60)
    )

    return fig

In [ ]:
# %run _bootstrap_dev.ipynb

# scelgo la frequenza e analizzo nel dettaglio il portfolio
# rebalance_freq='Y' # Frequenza di ribilanciamento, tipicamente annuale 
                     # oppure scelta in base al multifrequencies test 
rebalance_freq='Y'

# Date di analisi > lo start_date deve essere la data comune meno recente se si vuple un confronto realistico col benchmark (vedi normalize)

# start_date=common_start_date
# start_date='2011-10-20'
# start_date=None

# %run _bootstrap_dev.ipynb

final_portfolio  = run_portfolio_analysis(my_portfolio,
                                          title=my_portfolio_title,
                                          start_date=start_date,
                                          end_date=end_date,
                                          benchmark=benchmark,
                                          init_cash=init_cash,
                                          fees=0,
                                          rebalance_freq=rebalance_freq)




In [ ]:
# %run _bootstrap_dev.ipynb
f1 = visualize_portfolio_weights(final_portfolio,my_portfolio_title)
f1.show()

In [ ]:
# benchmark_portfolio = {
#     'SPY': 1.0,
#     'TLT': 0.0
# }

# final_benchmark_portfolio  = run_bh_backtest(benchmark_portfolio, 
#                                    start_date=start_date, 
#                                    end_date=end_date, 
#                                    init_cash=init_cash, 
#                                    fees=fees,
#                                    rebalance_freq=None)

# print(final_benchmark_portfolio.stats())
# final_benchmark_portfolio.plot()

In [ ]:
final_portfolio  = run_bh_backtest(my_portfolio, 
                                   start_date=start_date, 
                                   end_date=end_date, 
                                   init_cash=init_cash, 
                                   fees=fees,
                                   rebalance_freq=rebalance_freq)

asset_returns, port_return, port_curve = compute_portfolio_returns_pandas(my_portfolio, 
                                   start_date=start_date, 
                                   end_date=end_date)

In [ ]:
print("METODO VECTORBT")
print("▶️ RENDIMENTI PER ASSET")
print(final_portfolio.total_return(group_by=False))
print("▶️ RENDIMENTO PORTAFOGLIO")
print(f"{final_portfolio.total_return(group_by=True):.4%}")

print("\nMETODO PANDAS")
print("▶️ RENDIMENTI PER ASSET")
print(asset_returns.round(4))
print("▶️ RENDIMENTO PORTAFOGLIO")
print(f"{port_return:.4%}")

In [ ]:
# ticker = "VUAA.L"
# start = start_date
# end = end_date

# # Scarica i prezzi (1 solo ticker → indicizzazione singola)
# data = yf.download(ticker, start=start, end=end)
# adj_close = data["Close"].dropna()

# # Estraggo valori scalari reali
# price_start = adj_close.iloc[0].item() if hasattr(adj_close.iloc[0], 'item') else adj_close.iloc[0]
# price_end = adj_close.iloc[-1].item() if hasattr(adj_close.iloc[-1], 'item') else adj_close.iloc[-1]
# true_return = price_end / price_start - 1

# # Output
# print(f"▶️ Prezzo iniziale: {price_start:.2f}")
# print(f"▶️ Prezzo finale:   {price_end:.2f}")
# print(f"▶️ Rendimento reale: {true_return:.2%}")



In [ ]:
# # %run k_functions.ipynb
# res = efficient_frontier_pypfopt(
#     tickers=my_tickers,
#     my_weights=my_weights,
#     start_date=start_date,
#     # end_date='2025-01-01',    
#     end_date=end_date,
#     n_points=50,
#     weight_bounds=(0,1),
#     show_plot=True,
#     print_weights=True
# )



In [ ]:
# Portfolio efficiente. Verifica appartenenza alla frontiera efficiente

efficient_global_benchmark = {
    # "SP5A.MI": 0.0,  # Azionario sviluppato (S&P 500 UCITS)
    # "XMME.MI": 0.0,  # Mercati emergenti equity
    # "IBTM.MI": 0.0,  # Obbligazionario investment grade globale
    "IHYG.MI": 0.443,  # High yield globale
    "XAD5.MI": 0.373,  # Oro fisico
    # "TRET.MI": 0.00,  # Real estate globale
    "XLKS.MI": 0.0855,  # Settore tecnologico
    # "XLVS.MI": 0.00,  # Settore healthcare
    "XLFS.MI": 0.0985   # Settore finanziario
}

my_efficient_portfolio=efficient_global_benchmark
my_efficient_portfolio_title="Efficient Global Base"
my_efficient_tickets=list(my_efficient_portfolio.keys())
my_efficient_weights=list(my_efficient_portfolio.values())

years = 5

res = efficient_frontier_pypfopt(
    tickers=my_efficient_tickets,
    my_weights=my_efficient_weights,
    years=years,    
    n_points=50,
    weight_bounds=(0,1),
    show_plot=True,
    print_weights=True,
    compute_real_annual_return=True
)


In [ ]:
# # Portfolio efficiente
# rebalance_freq=None

# final_portfolio  = run_bh_backtest(my_efficient_portfolio, 
#                                    start_date=start_date, 
#                                    end_date=end_date, 
#                                    init_cash=init_cash, 
#                                    fees=fees,
#                                    rebalance_freq=rebalance_freq)
    
# # Stampa il risultato
# print("Backtest completato")
# print(f"\n--- Statistiche del Portafoglio {my_portfolio_title} (freq: {rebalance_freq}) ---")
# # print(final_portfolio.stats())
# final_portfolio.plot(group_by=True,width=vbt_plot_width).show()
# _ = print_summary(final_portfolio)

In [ ]:
years=10
benchmark = 'SPY'
# rebalance_freq='Y'
rebalance_freq=None

final_portfolio  = run_portfolio_analysis(my_efficient_portfolio,
                                          title=my_efficient_portfolio_title,
                                          # years=years,
                                          benchmark=benchmark,
                                          init_cash=init_cash,
                                          fees=fees,
                                          rebalance_freq=rebalance_freq)


In [ ]:
# optimization_result_sharpe = optimize_portfolio(
#     tickers=my_tickers,
#     start_date=start_date,
#     end_date=end_date,
#     target_metric='Sharpe Ratio',
#     goal='max',
#     num_trials=3000  # Aumentare per una ricerca più accurata
# )



In [ ]:
# optimization_result_sharpe = optimize_portfolio(
#     tickers=my_tickers,
#     start_date=start_date,
#     end_date=end_date,
#     target_metric='Total Return [%]',
#     goal='max',
#     num_trials=3000  # Aumentare per una ricerca più accurata
# )



## Core2 Benchmark

In [ ]:
my_portfolio=core2_benchmark
my_portfolio_title="Lazy Core2"

my_tickers=list(my_portfolio.keys())
my_weights=list(my_portfolio.values())

# periodo di analisi/backtest
start_date = '2019-01-01'
end_date = None
init_cash=10000
fees=0.0

In [ ]:
# Step 1: test frequenza di bilanciamento
rebalance_freqs=['W', 'M', 'Q', 'Y', None]

for rebalance_freq in rebalance_freqs:
    # Esegui il backtest 
    final_portfolio  = run_bh_backtest(my_portfolio, 
                                       start_date=start_date, 
                                       end_date=end_date, 
                                       init_cash=init_cash, 
                                       fees=fees,
                                       rebalance_freq=rebalance_freq)
    
    # Stampa il risultato
    print("Backtest completato")
    print(f"\n--- Statistiche del Portafoglio {my_portfolio_title} (freq: {rebalance_freq}) ---")
    # print(final_portfolio.stats())
    final_portfolio.plot(group_by=True,width=vbt_plot_width).show()
    _ = print_summary(final_portfolio)

In [ ]:
# scelgo la frequenza e analizzo nel dettaglio il portfolio
rebalance_freq=None # Scegli in base ai risultati precedente (rapporto rendimento/frequenza)
years=10
benchmark = 'SPY'

final_portfolio  = run_portfolio_analysis(my_portfolio,
                                          title=my_portfolio_title,
                                          years=years,
                                          benchmark=benchmark,
                                          init_cash=init_cash,
                                          fees=fees,
                                          rebalance_freq=rebalance_freq)

In [ ]:
res = efficient_frontier_pypfopt(
    tickers=my_tickers,
    my_weights=my_weights,
    start_date=start_date,
    # end_date='2025-01-01',    
    end_date=end_date,
    n_points=50,
    weight_bounds=(0,1),
    show_plot=True,
    print_weights=True
)



In [ ]:
efficient_core2_benchmark = {
    "AGGH": 0.0546,     # 🏦 Obbligazionario Governativo Globale (Euro)
    "HYLD.MI": 0.2141,  # 💳 Obbligazionario High Yield Globale
    "SWDA.MI": 0.2246,  # 🌍 Azionario Globale Paesi Sviluppati
    "SGLD.MI": 0.5067,  # 🥇 Oro fisico
}
my_efficient_portfolio=efficient_core2_benchmark
my_efficient_portfolio_title="Efficient Lazy Core2"

my_efficient_tickers=list(my_efficient_portfolio.keys())
my_efficient_weights=list(my_efficient_portfolio.values())

res = efficient_frontier_pypfopt(
    tickers=my_efficient_tickers,
    my_weights=my_efficient_weights,
    start_date=start_date,
    # end_date='2025-01-01',    
    end_date=end_date,
    n_points=50,
    weight_bounds=(0,1),
    show_plot=True,
    print_weights=True
)



In [ ]:
years=10
benchmark = 'SPY'
# rebalance_freq='Y'
rebalance_freq=None

final_portfolio  = run_portfolio_analysis(my_efficient_portfolio,
                                          title=my_efficient_portfolio_title,
                                          years=years,
                                          benchmark=benchmark,
                                          init_cash=init_cash,
                                          fees=fees,
                                          rebalance_freq=rebalance_freq)


## Gretchen

In [ ]:
my_portfolio=greta_hy_portfolio
my_portfolio_title="Gretchen HY"

# my_portfolio=greta_hy_portfolio
# my_portfolio_title="Gretchen HY"

# my_portfolio=greta_base_bitcoin_portfolio
# my_portfolio_title="Gretchen Base Bitcoin"

my_tickers=list(my_portfolio.keys())
my_weights=list(my_portfolio.values())

# periodo di analisi/backtest
start_date = '2019-01-01'
end_date = None
init_cash=10000
fees=0.0

In [ ]:
# Step 1: test frequenza di bilanciamento
rebalance_freqs=['W', 'M', 'Q', 'Y', None]

for rebalance_freq in rebalance_freqs:
    # Esegui il backtest 
    final_portfolio  = run_bh_backtest(my_portfolio, 
                                       start_date=start_date, 
                                       end_date=end_date, 
                                       init_cash=init_cash, 
                                       fees=fees,
                                       rebalance_freq=rebalance_freq)
    
    # Stampa il risultato
    print("Backtest completato")
    print(f"\n--- Statistiche del Portafoglio {my_portfolio_title} (freq: {rebalance_freq}) ---")
    # print(final_portfolio.stats())
    final_portfolio.plot(group_by=True,width=vbt_plot_width).show()
    _ = print_summary(final_portfolio)

In [ ]:
# scelgo la frequenza e analizzo nel dettaglio il portfolio
rebalance_freq=None # Scegli in base ai risultati precedente (rapporto rendimento/frequenza)
years=10
benchmark = 'SPY'

final_portfolio  = run_portfolio_analysis(my_portfolio,
                                          title=my_portfolio_title,
                                          years=years,
                                          benchmark=benchmark,
                                          init_cash=init_cash,
                                          fees=fees,
                                          rebalance_freq=rebalance_freq)

In [ ]:
res = efficient_frontier_pypfopt(
    tickers=my_tickers,
    my_weights=my_weights,
    start_date=start_date,
    # end_date='2025-01-01',    
    end_date=end_date,
    n_points=50,
    weight_bounds=(0,1),
    show_plot=True,
    print_weights=True
)


## Analisi Fondi Zurich

In [ ]:
fondi_df = pd.read_csv(
    "FondiZurich.csv",            # <-- sostituisci con il nome reale del file
    sep=",",
    decimal=",",                    # gestisce i numeri con virgola come decimali
    dayfirst=True,                  # per interpretare date in formato gg/mm/aaaa
    parse_dates=["Data operazione", "Data quotazione"],
    dtype={
        "ISIN": str,
        "Strumento": str,
        "Posizione": str
    }
)
fondi_isin=list(fondi_df.ISIN)
# print(f"Numero fondi censiti: {len(fondi_isin)}")

fondi_data=yf.download(fondi_isin,start='2023-01-01').Close
fondi_data=fondi_data.ffill().dropna(axis=1, how='all').dropna()
fondi_tickers=fondi_data.columns
display(fondi_data.tail())
print(f"Numero fondi censiti: {len(fondi_isin)}")
print(f"Numero fondi trovati: {len(fondi_tickers)}")
missing_cols = [col for col in fondi_isin  if col not in fondi_tickers]
print("Fondi censiti senza dati validi:")
print(missing_cols)


In [ ]:
init_cash=100_000
years=10
benchmark = 'SPY'
rebalance_freq=None

for t in fondi_tickers:
    print(f"Analisi fondo {BOLD}{t}{RESET} ....")

    single_ticker = {
        t : 1.0
    }

    final_portfolio  = run_portfolio_analysis(single_ticker, 
                                       years=years,
                                       benchmark=benchmark,
                                       init_cash=init_cash, 
                                       fees=fees,
                                       rebalance_freq=rebalance_freq)
